# 🎨 Autonomous Executive PowerPoint Agent (Plan-First + Dynamic Layouts + Gemini Image + HITL + SQLite)

### Architecture & Key Capabilities
1. **Plan-First Presentation Blueprint (`plan_presentation_blueprint`)**:
   - Before generating any image or slide, the agent **plans the entire presentation blueprint** slide-by-slide: selecting the narrative arc, section tags, layout types (`title`, `image_right`, `image_left`, `table`, `cards`, `ending`), and exact visual prompts.
2. **Dynamic, Non-Repetitive Slide Layouts (No Fixed Right-Only Images!)**:
   - **`title`**: Full-bleed Opening Hero Slide (`bg_opening_slide.png`) with a glassmorphism title card, neon pill badge, and metadata footer.
   - **`image_right`**: Stacked **Executive Feature Cards** on the **Left** + Framed Gemini Illustration on the **Right** + Bottom Key Takeaway Banner.
   - **`image_left`**: Framed Gemini Illustration on the **Left** + Stacked **Executive Feature Cards** on the **Right** + Bottom Key Takeaway Banner.
   - **`table`**: Native styled `python-pptx` **Comparison Table** (`slide.shapes.add_table`) with indigo/cyan header cells and alternating dark-slate rows **PLUS** a companion Gemini comparison visual diagram!
   - **`cards`**: Multi-card executive breakdown on the left + high-impact **Strategic Insight Panel** on the right.
   - **`ending`**: Closing Summary & Q&A Slide (`bg_closing_slide.png`) with numbered takeaway cards and an executive summary panel.
3. **Executive Card-Based Typography**:
   - Every bullet point (`"Concept Title: Detailed explanation"`) is rendered inside its own **elevated dark-slate rounded card** with a **left neon accent bar**, **numbered badge (`01`, `02`, `03`)**, **14pt Bold Snow-White heading**, and **11.5pt Bright Silver-White description**.
4. **Google Gemini 2.5 Flash Image (`gemini-2.5-flash-image`)**:
   - Generates custom 16:9 illustrations and visual comparison diagrams directly via `google-genai`.
5. **Human-in-the-Loop Middleware (`HumanInTheLoopMiddleware`) + Persistent `SqliteSaver`**:
   - Pauses at `submit_presentation_for_review` so you can inspect the `.pptx` draft and provide iterative enhancement feedback (`Command(resume=...)`).

In [ ]:
import os
import io
import json
import sqlite3
from typing import List, Dict, Optional, Any, Literal
from dotenv import load_dotenv

# Load environment variables (expects GOOGLE_API_KEY in .env)
load_dotenv()
load_dotenv("/Users/sachinmishra/Desktop/Agents_From_Scratch/.env")

# LangChain & LangGraph Core Imports
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command
from pydantic import BaseModel, Field

# Initialize Primary Reasoning Model (Google Gemini 3.1 Flash Lite)
model = init_chat_model("google_genai:gemini-3.1-flash-lite", temperature=0.2)
print("Initialized Primary Reasoning LLM: google_genai:gemini-3.1-flash-lite")

Initialized Primary Reasoning LLM: google_genai:gemini-2.5-flash


---
## 1. Progressive Disclosure Skill Loader (`load_skill`)
The agent loads the **Executive PPT Design & Theme Formatting Skill** (`skills/ppt_design/SKILL.md`) on demand to learn the mandatory **Plan -> Generate Images -> Compile -> Review** workflow, dynamic layout rules (`image_left`, `image_right`, `table`, `cards`), and card-based typography system.

In [2]:
SKILLS_DIR = os.path.join(os.getcwd(), "skills")


@tool
def load_skill(skill_name: str) -> str:
    """
    Loads the full design instructions and guidelines from a specialized skill folder.
    Use skill_name='ppt_design' to load the Executive PowerPoint Design & Layout rules.
    """
    skill_path = os.path.join(SKILLS_DIR, skill_name, "SKILL.md")
    if not os.path.exists(skill_path):
        return f"Skill '{skill_name}' not found at {skill_path}"
    with open(skill_path, "r", encoding="utf-8") as f:
        content = f.read()
    print(f"Loaded skill '{skill_name}' ({len(content):,} chars) from {skill_path}")
    return content

---
## 2. Step 1 Tool: Mandatory Presentation Blueprint Planner (`plan_presentation_blueprint`)
Before generating any images or slides, the agent **must** plan the complete deck structure using `plan_presentation_blueprint`.
- Ensures images are **not** always placed on the right-hand side (`image_right` vs. `image_left` vs. `table` vs. `cards`).
- Explicitly identifies which slides need custom visuals (`needs_image=True`) and defines their `visual_prompt` before calling `generate_slide_image`.

In [3]:
class SlideBlueprintItem(BaseModel):
    slide_number: int = Field(description="1-based slide index in the deck.")
    section_tag: str = Field(description="Short uppercase section label, e.g. '01 // OVERVIEW' or '03 // COMPARISON'.")
    title: str = Field(description="Slide headline title.")
    layout: Literal["title", "image_right", "image_left", "table", "cards", "ending"] = Field(
        description=(
            "Slide layout type. MUST vary across the deck: "
            "'title' (opening), 'image_right' (cards left + image right), "
            "'image_left' (image left + cards right), 'table' (styled comparison table + visual chart), "
            "'cards' (multi-card grid + strategic insight panel), 'ending' (closing summary)."
        )
    )
    needs_image: bool = Field(
        description="True if this slide requires a Gemini-generated visual (True for 'image_right', 'image_left', and 'table')."
    )
    image_position: Literal["right", "left", "table_companion", "none"] = Field(
        description="Where the visual will sit on the slide ('right', 'left', 'table_companion', or 'none')."
    )
    visual_prompt: Optional[str] = Field(
        default=None,
        description="Detailed prompt for generate_slide_image if needs_image is True.",
    )
    planned_key_points: List[str] = Field(
        description="Brief list of the 3-4 core concepts or table columns planned for this slide."
    )


@tool
def plan_presentation_blueprint(
    topic: str,
    target_audience: str,
    slide_plans: List[SlideBlueprintItem],
) -> str:
    """
    MANDATORY STEP 1: Plans the complete slide-by-slide blueprint BEFORE generating any images or compiling the PPTX.
    Ensures dynamic layout variety (alternating 'image_right', 'image_left', 'table', 'cards', 'title', 'ending')
    and defines which slides require Gemini image generation.
    """
    os.makedirs("generated_Blueprint_Plans", exist_ok=True)
    plan_data = {
        "topic": topic,
        "target_audience": target_audience,
        "total_slides": len(slide_plans),
        "slides": [s.model_dump() if hasattr(s, "model_dump") else dict(s) for s in slide_plans],
    }
    plan_path = os.path.join("generated_Blueprint_Plans", "latest_presentation_blueprint.json")
    with open(plan_path, "w", encoding="utf-8") as f:
        json.dump(plan_data, f, indent=2)

    layout_summary = [
        f"Slide {s['slide_number']}: [{s['layout'].upper()}] {s['title']} (Image: {s['image_position']})"
        for s in plan_data["slides"]
    ]
    image_slides = [s for s in plan_data["slides"] if s.get("needs_image")]

    print("\n" + "=" * 75)
    print(f"PRESENTATION BLUEPRINT LOCKED: {topic} ({len(slide_plans)} Slides)")
    for line in layout_summary:
        print("  •", line)
    print(f"  -> Planned Gemini Visuals to Generate: {len(image_slides)} slide(s)")
    print("=" * 75 + "\n")

    return (
        f"Presentation Blueprint approved and saved to '{plan_path}'.\n"
        f"Layout sequence:\n" + "\n".join(layout_summary) + "\n\n"
        f"NEXT STEP: Call `generate_slide_image` for the {len(image_slides)} planned visual slide(s), "
        f"then compile the deck with `create_premium_presentation`."
    )

---
## 3. Step 2 Tool: Gemini 2.5 Flash Image Generator (`generate_slide_image`)
Uses the `google-genai` SDK (`gemini-2.5-flash-image` with `response_modalities=["IMAGE"]`) to generate custom 16:9 scientific illustrations, architectural diagrams, and visual comparison charts according to the blueprint.

In [4]:
def _gemini_generate_image_bytes(prompt: str) -> bytes:
    """
    Calls Google GenAI SDK (google-genai) to generate image bytes using 'gemini-2.5-flash-image'.
    Returns raw binary PNG/JPEG bytes.
    """
    from google import genai
    from google.genai import types

    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError("GOOGLE_API_KEY is not set.")

    client = genai.Client(api_key=api_key)

    resp = client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_modalities=["IMAGE"],  # Requests binary image output instead of text
            safety_settings=[
                types.SafetySetting(
                    category="HARM_CATEGORY_DANGEROUS_CONTENT",
                    threshold="BLOCK_ONLY_HIGH",
                )
            ],
        ),
    )

    # Extract raw image bytes from candidates -> content -> parts -> inline_data
    for cand in getattr(resp, "candidates", []) or []:
        content = getattr(cand, "content", None)
        if not content:
            continue
        for part in getattr(content, "parts", []) or []:
            inline = getattr(part, "inline_data", None)
            if inline and getattr(inline, "data", None):
                data = inline.data
                return data if isinstance(data, (bytes, bytearray)) else bytes(data)

    raise RuntimeError("gemini-2.5-flash-image returned no inline image bytes.")


@tool
def generate_slide_image(prompt: str, slide_number: int, visual_type: str = "illustration") -> str:
    """
    Generates a high-resolution visual using Google's 'gemini-2.5-flash-image' model.
    - prompt: Detailed visual description from the presentation blueprint.
    - slide_number: Slide number this image belongs to.
    - visual_type: 'illustration', 'diagram', or 'table_infographic'.
    Returns the local file path to the saved PNG image.
    """
    output_dir = os.path.join(os.getcwd(), "generated_Slide_Images")
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, f"slide_{slide_number}_{visual_type}.png")

    style_suffix = (
        " | Style: Ultra-clean modern executive scientific visualization, dark slate/navy background (#0F172A), "
        "bioluminescent cyan (#00F2FE), royal purple (#A855F7), and emerald (#10B981) highlights, "
        "crisp vector-grade clarity, 16:9 aspect ratio, high contrast."
    )
    if visual_type == "table_infographic":
        style_suffix = (
            " | Style: Executive dark-mode visual comparison infographic matrix, structured side-by-side "
            "comparison blocks with glowing cyan (#00F2FE) and purple (#A855F7) icons and visual indicators, "
            "deep navy background (#0F172A), clean modern UI aesthetic."
        )

    enhanced_prompt = prompt + style_suffix
    print(f"Generating Gemini 2.5 Flash Image for Slide {slide_number} ({visual_type})...")
    img_bytes = _gemini_generate_image_bytes(enhanced_prompt)
    with open(file_path, "wb") as f:
        f.write(img_bytes)
    print(f"Saved Slide {slide_number} image -> {file_path} ({len(img_bytes):,} bytes)")
    return file_path

---
## 4. Step 3 Tool: Executive PowerPoint Layout & Typography Engine (`create_premium_presentation`)

### Key Upgrades in This Engine:
1. **Executive Card-Based Typography (`_render_feature_cards_column`)**:
   - Instead of plain floating bullet points (`• text`) in a bare text box, every bullet (`"Lead Title: Detailed explanation"`) is rendered inside its own **Elevated Dark-Slate Rounded Card** (`MSO_SHAPE.ROUNDED_RECTANGLE`) with:
     - A **vertical neon accent bar** (`MSO_SHAPE.RECTANGLE`) on the left edge (cycling Cyan, Purple, Emerald, Gold).
     - A **numbered badge (`01`, `02`, `03`, `04`)** in a rounded badge box.
     - A **`14pt Bold` Snow-White heading** (`RGB(248, 250, 252)`) on top.
     - A **`11.5pt` Bright Silver-White description** (`RGB(214, 224, 238)`) directly underneath.
2. **Flexible Left & Right Image Positioning (`_render_framed_image_panel`)**:
   - **`layout="image_right"`**: Feature Cards on Left (`x=0.75"`) + Framed Gemini Visual on Right (`x=6.95"`).
   - **`layout="image_left"`**: Framed Gemini Visual on Left (`x=0.75"`) + Feature Cards on Right (`x=6.63"`).
3. **Native Styled Comparison Table + Table Visual (`layout="table"`)**:
   - Uses `slide.shapes.add_table(rows, cols, left, top, width, height)` to build a dark-mode executive comparison table with styled header cells and alternating slate rows.
   - If `image_path` is also provided on the `"table"` slide, embeds the companion Gemini comparison infographic alongside the native table!
4. **Full-Width Bottom Key Takeaway Strip (`_render_bottom_takeaway_bar`)**:
   - Anchors the bottom of every content slide (`top=6.42"`) with a high-contrast `KEY INSIGHT` banner so slides look complete and balanced.

In [5]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE

# ==============================================================================
# EXECUTIVE COLOR PALETTE & FONT SYSTEM
# ==============================================================================
BG_DARK_SLATE     = RGBColor(9, 13, 24)       # Deep Obsidian Canvas Base (#090D18)
CARD_BG           = RGBColor(20, 30, 52)      # Elevated Slate Feature Card (#141E34)
CARD_BG_ALT       = RGBColor(26, 39, 66)      # Slightly Lighter Slate Card (#1A2742)
CARD_BORDER       = RGBColor(48, 72, 115)     # Crisp Slate Hairline Border (#304873)
FRAME_BG          = RGBColor(14, 22, 38)      # Visual Frame Container Fill (#0E1626)

TEXT_WHITE        = RGBColor(248, 250, 252)   # Pure Snow White for Titles & Card Headers
TEXT_BRIGHT_BODY  = RGBColor(214, 224, 238)   # Bright Silver-White for High-Contrast Body Text
TEXT_SUBTITLE     = RGBColor(165, 195, 225)   # Soft Ice-Blue for Subtitles & Captions

ACCENT_CYAN       = RGBColor(0, 242, 254)     # Neon Cyan (#00F2FE)
ACCENT_PURPLE     = RGBColor(168, 85, 247)    # Electric Royal Purple (#A855F7)
ACCENT_EMERALD    = RGBColor(16, 185, 129)    # Vibrant Emerald (#10B981)
ACCENT_GOLD       = RGBColor(245, 158, 11)    # Executive Amber Gold (#F59E0B)

ACCENT_PALETTE    = [ACCENT_CYAN, ACCENT_PURPLE, ACCENT_EMERALD, ACCENT_GOLD]
FONT_HEADING      = "Calibri"
FONT_BODY         = "Calibri"

RESOURCES_DIR = os.path.join(SKILLS_DIR, "ppt_design", "resources")
BG_IMAGE_MAP = {
    "title":       os.path.join(RESOURCES_DIR, "bg_opening_slide.png"),
    "image_right": os.path.join(RESOURCES_DIR, "bg_content_image.png"),
    "image_left":  os.path.join(RESOURCES_DIR, "bg_content_image.png"),
    "image":       os.path.join(RESOURCES_DIR, "bg_content_image.png"),
    "table":       os.path.join(RESOURCES_DIR, "bg_content_text.png"),
    "cards":       os.path.join(RESOURCES_DIR, "bg_content_text.png"),
    "text":        os.path.join(RESOURCES_DIR, "bg_content_text.png"),
    "ending":      os.path.join(RESOURCES_DIR, "bg_closing_slide.png"),
}


def _apply_slide_background(slide, prs, layout_type: str):
    """
    Applies the solid dark fallback color AND places the corresponding 16:9 executive
    background PNG across the full slide canvas (13.333" x 7.5") as the bottom layer.
    """
    bg = slide.background
    fill = bg.fill
    fill.solid()
    fill.fore_color.rgb = BG_DARK_SLATE

    bg_path = BG_IMAGE_MAP.get(layout_type, BG_IMAGE_MAP["cards"])
    if os.path.exists(bg_path):
        slide.shapes.add_picture(
            bg_path,
            left=Inches(0),
            top=Inches(0),
            width=prs.slide_width,
            height=prs.slide_height,
        )


def _split_bullet_title_and_body(bullet_text: str):
    """
    Splits a bullet string formatted as 'Bold Concept Title: Detailed explanation...'
    into (title_part, body_part). If no colon is present, extracts the first few words
    as the bold heading so EVERY card has a crisp two-tier typographic hierarchy.
    """
    clean = bullet_text.strip().lstrip("•-* ").strip()
    if ":" in clean:
        parts = clean.split(":", 1)
        title_part = parts[0].strip()
        body_part = parts[1].strip()
        if title_part and body_part:
            return title_part, body_part
    words = clean.split()
    if len(words) > 5:
        return " ".join(words[:4]), clean
    return clean, ""


def _render_executive_header(
    slide,
    title_text: str,
    subtitle_text: str,
    section_tag: str,
    slide_idx: int,
    total_slides: int,
    accent_color: RGBColor = ACCENT_CYAN,
):
    """
    Renders a consistent, high-contrast executive header on content slides:
    - Top-left rounded pill badge with the uppercase section tag
    - Top-right slide counter pill badge ('SLIDE 02 / 06')
    - 25pt Bold Snow-White main headline
    - 12.5pt Soft Ice-Blue subtitle lead-in
    - Dual-tone glowing divider bar at top=1.56"
    """
    # 1. Top-Left Section Pill Badge
    pill = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(0.75), Inches(0.32), Inches(3.45), Inches(0.30)
    )
    pill.fill.solid()
    pill.fill.fore_color.rgb = RGBColor(16, 28, 52)
    pill.line.color.rgb = accent_color
    pill.line.width = Pt(1.2)
    tf_pill = pill.text_frame
    tf_pill.word_wrap = False
    tf_pill.margin_left = Inches(0.12)
    tf_pill.margin_top = Inches(0.03)
    p_pill = tf_pill.paragraphs[0]
    p_pill.text = section_tag.upper()
    p_pill.font.name = FONT_HEADING
    p_pill.font.size = Pt(9.5)
    p_pill.font.bold = True
    p_pill.font.color.rgb = accent_color

    # 2. Top-Right Slide Number Pill Badge
    num_pill = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(11.43), Inches(0.32), Inches(1.15), Inches(0.30)
    )
    num_pill.fill.solid()
    num_pill.fill.fore_color.rgb = RGBColor(16, 28, 52)
    num_pill.line.color.rgb = CARD_BORDER
    num_pill.line.width = Pt(1.0)
    tf_num = num_pill.text_frame
    tf_num.margin_top = Inches(0.03)
    p_num = tf_num.paragraphs[0]
    p_num.alignment = PP_ALIGN.CENTER
    p_num.text = f"SLIDE {slide_idx:02d} / {total_slides:02d}"
    p_num.font.name = FONT_HEADING
    p_num.font.size = Pt(9)
    p_num.font.bold = True
    p_num.font.color.rgb = TEXT_SUBTITLE

    # 3. Main Slide Title (25pt Bold Snow White)
    title_box = slide.shapes.add_textbox(Inches(0.75), Inches(0.66), Inches(11.83), Inches(0.50))
    tf_title = title_box.text_frame
    tf_title.word_wrap = True
    tf_title.margin_left = Inches(0)
    tf_title.margin_top = Inches(0)
    p_title = tf_title.paragraphs[0]
    p_title.text = title_text
    p_title.font.name = FONT_HEADING
    p_title.font.size = Pt(25)
    p_title.font.bold = True
    p_title.font.color.rgb = TEXT_WHITE

    # 4. Subtitle / Narrative Lead-in (12.5pt Ice-Blue)
    if subtitle_text:
        sub_box = slide.shapes.add_textbox(Inches(0.75), Inches(1.17), Inches(11.83), Inches(0.34))
        tf_sub = sub_box.text_frame
        tf_sub.word_wrap = True
        tf_sub.margin_left = Inches(0)
        tf_sub.margin_top = Inches(0)
        p_sub = tf_sub.paragraphs[0]
        p_sub.text = subtitle_text
        p_sub.font.name = FONT_BODY
        p_sub.font.size = Pt(12.5)
        p_sub.font.color.rgb = TEXT_SUBTITLE

    # 5. Dual-Tone Header Divider Line
    line_accent = slide.shapes.add_shape(
        MSO_SHAPE.RECTANGLE, Inches(0.75), Inches(1.56), Inches(3.40), Inches(0.035)
    )
    line_accent.fill.solid()
    line_accent.fill.fore_color.rgb = accent_color
    line_accent.line.fill.background()

    line_rest = slide.shapes.add_shape(
        MSO_SHAPE.RECTANGLE, Inches(4.15), Inches(1.57), Inches(8.43), Inches(0.015)
    )
    line_rest.fill.solid()
    line_rest.fill.fore_color.rgb = CARD_BORDER
    line_rest.line.fill.background()


def _render_feature_cards_column(
    slide,
    bullets: List[str],
    col_left: float,
    col_top: float,
    col_width: float,
    col_height: float,
):
    """
    Renders a vertical stack of sleek Executive Feature Cards instead of plain bullets.
    Each card includes:
    - Rounded dark-slate container box with hairline border
    - Left vertical neon accent strip (cycling Cyan, Purple, Emerald, Gold)
    - Numbered badge ('01', '02', '03'...)
    - Bold 14pt Snow-White Title + Crisp 11.5pt Bright Silver-White Description
    """
    items = bullets[:4] if bullets else ["Key Point: Details provided in presentation."]
    n = len(items)
    gap = 0.14 if n <= 3 else 0.10
    card_h = (col_height - (n - 1) * gap) / n

    for idx, raw_bullet in enumerate(items):
        accent = ACCENT_PALETTE[idx % len(ACCENT_PALETTE)]
        curr_top = col_top + idx * (card_h + gap)
        heading_str, body_str = _split_bullet_title_and_body(raw_bullet)

        # 1. Rounded Card Background Container
        card = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE,
            Inches(col_left), Inches(curr_top), Inches(col_width), Inches(card_h)
        )
        card.fill.solid()
        card.fill.fore_color.rgb = CARD_BG if idx % 2 == 0 else CARD_BG_ALT
        card.line.color.rgb = CARD_BORDER
        card.line.width = Pt(1.2)

        # 2. Glowing Left Vertical Accent Strip inside Card
        strip = slide.shapes.add_shape(
            MSO_SHAPE.RECTANGLE,
            Inches(col_left + 0.06), Inches(curr_top + 0.10), Inches(0.07), Inches(card_h - 0.20)
        )
        strip.fill.solid()
        strip.fill.fore_color.rgb = accent
        strip.line.fill.background()

        # 3. Numbered Badge ('01', '02', ...)
        badge_h = min(0.42, card_h - 0.20)
        badge = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE,
            Inches(col_left + 0.22), Inches(curr_top + 0.14), Inches(0.46), Inches(badge_h)
        )
        badge.fill.solid()
        badge.fill.fore_color.rgb = RGBColor(11, 18, 34)
        badge.line.color.rgb = accent
        badge.line.width = Pt(1.2)
        tf_b = badge.text_frame
        tf_b.margin_left = Inches(0.02)
        tf_b.margin_right = Inches(0.02)
        tf_b.margin_top = Inches(0.05)
        p_b = tf_b.paragraphs[0]
        p_b.alignment = PP_ALIGN.CENTER
        p_b.text = f"{idx + 1:02d}"
        p_b.font.name = FONT_HEADING
        p_b.font.size = Pt(11)
        p_b.font.bold = True
        p_b.font.color.rgb = accent

        # 4. Two-Tier Card Typography (Bold White Heading + Bright Silver Body)
        tb = slide.shapes.add_textbox(
            Inches(col_left + 0.78),
            Inches(curr_top + 0.08),
            Inches(col_width - 0.90),
            Inches(card_h - 0.16),
        )
        tf = tb.text_frame
        tf.word_wrap = True
        tf.margin_left = Inches(0.04)
        tf.margin_right = Inches(0.06)
        tf.margin_top = Inches(0.03)
        tf.margin_bottom = Inches(0.03)

        p_head = tf.paragraphs[0]
        p_head.text = heading_str
        p_head.font.name = FONT_HEADING
        p_head.font.size = Pt(14 if n <= 3 else 13)
        p_head.font.bold = True
        p_head.font.color.rgb = TEXT_WHITE
        p_head.space_after = Pt(3)

        if body_str:
            p_body = tf.add_paragraph()
            p_body.text = body_str
            p_body.font.name = FONT_BODY
            p_body.font.size = Pt(11.5 if n <= 3 else 10.8)
            p_body.font.color.rgb = TEXT_BRIGHT_BODY


def _render_framed_image_panel(
    slide,
    image_path: Optional[str],
    panel_left: float,
    panel_top: float,
    panel_width: float,
    panel_height: float,
    caption_text: str,
    accent_color: RGBColor = ACCENT_CYAN,
    badge_label: str = "GEMINI 2.5 FLASH VISUAL",
):
    """
    Renders an elevated dark-slate frame at ANY position (Left or Right or Table Companion)
    and embeds the Gemini-generated illustration inside with a top header bar and bottom caption.
    """
    # 1. Outer Elevated Frame Card
    frame = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(panel_left), Inches(panel_top), Inches(panel_width), Inches(panel_height)
    )
    frame.fill.solid()
    frame.fill.fore_color.rgb = FRAME_BG
    frame.line.color.rgb = accent_color
    frame.line.width = Pt(1.5)

    # 2. Top Frame Header Strip
    hdr_box = slide.shapes.add_textbox(
        Inches(panel_left + 0.16), Inches(panel_top + 0.06), Inches(panel_width - 0.32), Inches(0.28)
    )
    tf_hdr = hdr_box.text_frame
    tf_hdr.margin_left = Inches(0)
    tf_hdr.margin_top = Inches(0.02)
    p_hdr = tf_hdr.paragraphs[0]
    p_hdr.text = f"◆  {badge_label.upper()}"
    p_hdr.font.name = FONT_HEADING
    p_hdr.font.size = Pt(9.5)
    p_hdr.font.bold = True
    p_hdr.font.color.rgb = accent_color

    # 3. Embedded High-Res PNG Illustration
    img_left = panel_left + 0.14
    img_top = panel_top + 0.36
    img_w = panel_width - 0.28
    img_h = panel_height - 0.86

    if image_path and os.path.exists(image_path):
        slide.shapes.add_picture(
            image_path,
            left=Inches(img_left),
            top=Inches(img_top),
            width=Inches(img_w),
            height=Inches(img_h),
        )
    else:
        placeholder = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE,
            Inches(img_left), Inches(img_top), Inches(img_w), Inches(img_h)
        )
        placeholder.fill.solid()
        placeholder.fill.fore_color.rgb = CARD_BG
        placeholder.line.color.rgb = CARD_BORDER

    # 4. Bottom Visual Caption Bar inside Frame
    cap_box = slide.shapes.add_textbox(
        Inches(panel_left + 0.16),
        Inches(panel_top + panel_height - 0.46),
        Inches(panel_width - 0.32),
        Inches(0.40),
    )
    tf_cap = cap_box.text_frame
    tf_cap.word_wrap = True
    tf_cap.margin_left = Inches(0)
    tf_cap.margin_top = Inches(0.02)
    p_cap = tf_cap.paragraphs[0]
    p_cap.text = caption_text[:135] if caption_text else "AI-generated architectural visual aligned with slide blueprint."
    p_cap.font.name = FONT_BODY
    p_cap.font.size = Pt(10)
    p_cap.font.color.rgb = TEXT_SUBTITLE


def _render_bottom_takeaway_bar(
    slide,
    takeaway_text: str,
    accent_color: RGBColor = ACCENT_CYAN,
):
    """
    Renders a full-width horizontal 'KEY INSIGHT' banner across the bottom of a content slide
    (left=0.75", top=6.42", width=11.83", height=0.68") so the slide feels anchored and complete.
    """
    if not takeaway_text:
        return
    banner = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(0.75), Inches(6.42), Inches(11.83), Inches(0.68)
    )
    banner.fill.solid()
    banner.fill.fore_color.rgb = RGBColor(15, 24, 44)
    banner.line.color.rgb = accent_color
    banner.line.width = Pt(1.2)

    # Left Badge inside Banner
    tag = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(0.90), Inches(6.55), Inches(1.65), Inches(0.42)
    )
    tag.fill.solid()
    tag.fill.fore_color.rgb = accent_color
    tag.line.fill.background()
    tf_t = tag.text_frame
    tf_t.margin_top = Inches(0.08)
    p_t = tf_t.paragraphs[0]
    p_t.alignment = PP_ALIGN.CENTER
    p_t.text = "KEY INSIGHT"
    p_t.font.name = FONT_HEADING
    p_t.font.size = Pt(9.5)
    p_t.font.bold = True
    p_t.font.color.rgb = BG_DARK_SLATE

    # Takeaway Description Text
    tb = slide.shapes.add_textbox(Inches(2.70), Inches(6.48), Inches(9.70), Inches(0.56))
    tf = tb.text_frame
    tf.word_wrap = True
    tf.margin_left = Inches(0)
    tf.margin_top = Inches(0.04)
    p = tf.paragraphs[0]
    p.text = takeaway_text
    p.font.name = FONT_BODY
    p.font.size = Pt(11.5)
    p.font.bold = True
    p.font.color.rgb = TEXT_WHITE


def _render_styled_table(
    slide,
    table_data: Dict[str, Any],
    left: float,
    top: float,
    width: float,
    height: float,
):
    """
    Creates a native PowerPoint table (`slide.shapes.add_table`) styled with an executive
    dark-mode theme: Royal Indigo/Cyan header cells, bold white headers, and alternating
    dark-slate data rows with high-contrast typography.
    """
    headers = table_data.get("headers", ["Dimension", "Innate Immunity", "Adaptive Immunity"])
    rows = table_data.get("rows", [])
    num_rows = len(rows) + 1
    num_cols = len(headers)

    # Background frame behind table for crisp border styling
    t_frame = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(left - 0.06), Inches(top - 0.06), Inches(width + 0.12), Inches(height + 0.12)
    )
    t_frame.fill.solid()
    t_frame.fill.fore_color.rgb = FRAME_BG
    t_frame.line.color.rgb = ACCENT_CYAN
    t_frame.line.width = Pt(1.4)

    table_shape = slide.shapes.add_table(
        num_rows, num_cols, Inches(left), Inches(top), Inches(width), Inches(height)
    )
    table = table_shape.table

    # Style Header Row
    for c_idx, h_text in enumerate(headers):
        cell = table.cell(0, c_idx)
        cell.fill.solid()
        cell.fill.fore_color.rgb = RGBColor(26, 49, 94)
        cell.vertical_anchor = MSO_ANCHOR.MIDDLE
        tf = cell.text_frame
        tf.word_wrap = True
        p = tf.paragraphs[0]
        p.text = str(h_text)
        p.font.name = FONT_HEADING
        p.font.size = Pt(12)
        p.font.bold = True
        p.font.color.rgb = ACCENT_CYAN if c_idx == 0 else TEXT_WHITE

    # Style Data Rows
    for r_idx, row_values in enumerate(rows):
        row_bg = RGBColor(18, 28, 50) if r_idx % 2 == 0 else RGBColor(25, 38, 66)
        for c_idx in range(num_cols):
            val = row_values[c_idx] if c_idx < len(row_values) else ""
            cell = table.cell(r_idx + 1, c_idx)
            cell.fill.solid()
            cell.fill.fore_color.rgb = row_bg
            cell.vertical_anchor = MSO_ANCHOR.MIDDLE
            tf = cell.text_frame
            tf.word_wrap = True
            p = tf.paragraphs[0]
            p.text = str(val)
            p.font.name = FONT_BODY
            p.font.size = Pt(11)
            p.font.bold = (c_idx == 0)
            p.font.color.rgb = ACCENT_GOLD if c_idx == 0 else TEXT_BRIGHT_BODY


@tool
def create_premium_presentation(slides_data: List[Dict[str, Any]], file_name: str = "presentation.pptx") -> str:
    """
    Compiles a professional 16:9 PowerPoint presentation (.pptx) with executive card-based
    typography and 6 dynamic slide layouts:
      - 'title': Opening Hero Slide (bg_opening_slide.png)
      - 'image_right': Feature Cards on LEFT + Framed Gemini Illustration on RIGHT
      - 'image_left': Framed Gemini Illustration on LEFT + Feature Cards on RIGHT
      - 'table': Native Styled Comparison Table (`table_data`) + Companion Gemini Table Visual (`image_path`)
      - 'cards' (or 'text'): Multi-Card Feature Breakdown + Strategic Insight Panel
      - 'ending': Closing Takeaways & Executive Summary Card (bg_closing_slide.png)

    Each slide dict in `slides_data` supports:
      - layout: 'title' | 'image_right' | 'image_left' | 'table' | 'cards' | 'ending'
      - section_tag: str (e.g. '02 // FIRST-LINE DEFENSE')
      - title: str
      - subtitle: Optional[str]
      - bullets: Optional[List[str]] (formatted as 'Bold Title: Detailed explanation')
      - key_insight: Optional[str] (bottom takeaway banner or side insight panel)
      - image_path: Optional[str] (local PNG path returned by generate_slide_image)
      - table_data: Optional[Dict[str, Any]] ({'headers': [...], 'rows': [[...], ...]} for layout='table')
    """
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    blank_layout = prs.slide_layouts[6]
    total_slides = len(slides_data)

    for idx, slide_info in enumerate(slides_data, start=1):
        layout_type = slide_info.get("layout", "cards").lower()
        if layout_type == "image":
            # Automatically alternate left/right if generic 'image' was passed
            layout_type = "image_right" if idx % 2 == 0 else "image_left"

        slide = prs.slides.add_slide(blank_layout)
        _apply_slide_background(slide, prs, layout_type)

        title_text = slide_info.get("title", "")
        subtitle_text = slide_info.get("subtitle", "")
        section_tag = slide_info.get("section_tag", f"SECTION 0{idx} // EXECUTIVE BRIEFING")
        bullets = slide_info.get("bullets", [])
        key_insight = slide_info.get("key_insight", "")
        image_path = slide_info.get("image_path")
        table_data = slide_info.get("table_data")

        # ======================================================================
        # LAYOUT 1: OPENING HERO SLIDE ('title')
        # ======================================================================
        if layout_type == "title":
            hero_card = slide.shapes.add_shape(
                MSO_SHAPE.ROUNDED_RECTANGLE,
                Inches(1.15), Inches(1.25), Inches(11.033), Inches(4.95)
            )
            hero_card.fill.solid()
            hero_card.fill.fore_color.rgb = RGBColor(14, 22, 40)
            hero_card.line.color.rgb = ACCENT_CYAN
            hero_card.line.width = Pt(1.8)

            top_bar = slide.shapes.add_shape(
                MSO_SHAPE.RECTANGLE,
                Inches(4.40), Inches(1.25), Inches(4.533), Inches(0.08)
            )
            top_bar.fill.solid()
            top_bar.fill.fore_color.rgb = ACCENT_CYAN
            top_bar.line.fill.background()

            badge = slide.shapes.add_shape(
                MSO_SHAPE.ROUNDED_RECTANGLE,
                Inches(4.45), Inches(1.72), Inches(4.433), Inches(0.40)
            )
            badge.fill.solid()
            badge.fill.fore_color.rgb = RGBColor(22, 36, 66)
            badge.line.color.rgb = ACCENT_PURPLE
            badge.line.width = Pt(1.2)
            tf_bg = badge.text_frame
            tf_bg.margin_top = Inches(0.06)
            p_bg = tf_bg.paragraphs[0]
            p_bg.alignment = PP_ALIGN.CENTER
            p_bg.text = section_tag.upper() if section_tag else "EXECUTIVE SCIENTIFIC BRIEFING"
            p_bg.font.name = FONT_HEADING
            p_bg.font.size = Pt(10.5)
            p_bg.font.bold = True
            p_bg.font.color.rgb = ACCENT_CYAN

            t_box = slide.shapes.add_textbox(Inches(1.65), Inches(2.35), Inches(10.033), Inches(1.75))
            tf_t = t_box.text_frame
            tf_t.word_wrap = True
            p_t = tf_t.paragraphs[0]
            p_t.alignment = PP_ALIGN.CENTER
            p_t.text = title_text
            p_t.font.name = FONT_HEADING
            p_t.font.size = Pt(36)
            p_t.font.bold = True
            p_t.font.color.rgb = TEXT_WHITE

            if subtitle_text:
                s_box = slide.shapes.add_textbox(Inches(2.05), Inches(4.18), Inches(9.233), Inches(1.05))
                tf_s = s_box.text_frame
                tf_s.word_wrap = True
                p_s = tf_s.paragraphs[0]
                p_s.alignment = PP_ALIGN.CENTER
                p_s.text = subtitle_text
                p_s.font.name = FONT_BODY
                p_s.font.size = Pt(16)
                p_s.font.color.rgb = TEXT_SUBTITLE

            meta_box = slide.shapes.add_textbox(Inches(2.05), Inches(5.45), Inches(9.233), Inches(0.45))
            tf_m = meta_box.text_frame
            p_m = tf_m.paragraphs[0]
            p_m.alignment = PP_ALIGN.CENTER
            p_m.text = f"AUTONOMOUS AI PRESENTATION ARCHITECTURE  •  {total_slides} SLIDES  •  GEMINI 2.5 FLASH VISUALS"
            p_m.font.name = FONT_HEADING
            p_m.font.size = Pt(10)
            p_m.font.bold = True
            p_m.font.color.rgb = ACCENT_GOLD

        # ======================================================================
        # LAYOUT 2: FEATURE CARDS LEFT + GEMINI IMAGE RIGHT ('image_right')
        # ======================================================================
        elif layout_type == "image_right":
            _render_executive_header(
                slide, title_text, subtitle_text, section_tag, idx, total_slides, ACCENT_CYAN
            )
            content_h = 4.48 if key_insight else 5.15
            _render_feature_cards_column(
                slide, bullets, col_left=0.75, col_top=1.74, col_width=5.95, col_height=content_h
            )
            _render_framed_image_panel(
                slide,
                image_path=image_path,
                panel_left=6.95,
                panel_top=1.74,
                panel_width=5.63,
                panel_height=content_h,
                caption_text=subtitle_text or title_text,
                accent_color=ACCENT_CYAN,
                badge_label="VISUAL ARCHITECTURE // RIGHT PANEL",
            )
            _render_bottom_takeaway_bar(slide, key_insight, ACCENT_CYAN)

        # ======================================================================
        # LAYOUT 3: GEMINI IMAGE LEFT + FEATURE CARDS RIGHT ('image_left')
        # ======================================================================
        elif layout_type == "image_left":
            _render_executive_header(
                slide, title_text, subtitle_text, section_tag, idx, total_slides, ACCENT_PURPLE
            )
            content_h = 4.48 if key_insight else 5.15
            _render_framed_image_panel(
                slide,
                image_path=image_path,
                panel_left=0.75,
                panel_top=1.74,
                panel_width=5.63,
                panel_height=content_h,
                caption_text=subtitle_text or title_text,
                accent_color=ACCENT_PURPLE,
                badge_label="VISUAL ARCHITECTURE // LEFT PANEL",
            )
            _render_feature_cards_column(
                slide, bullets, col_left=6.63, col_top=1.74, col_width=5.95, col_height=content_h
            )
            _render_bottom_takeaway_bar(slide, key_insight, ACCENT_PURPLE)

        # ======================================================================
        # LAYOUT 4: STYLED COMPARISON TABLE + TABLE VISUAL ('table')
        # ======================================================================
        elif layout_type == "table":
            _render_executive_header(
                slide, title_text, subtitle_text, section_tag, idx, total_slides, ACCENT_EMERALD
            )
            content_h = 4.48 if key_insight else 5.15

            if not table_data:
                rows_parsed = []
                for b in bullets[:5]:
                    h, d = _split_bullet_title_and_body(b)
                    rows_parsed.append([h, d])
                table_data = {
                    "headers": ["Dimension / Mechanism", "Key Functional Specification"],
                    "rows": rows_parsed,
                }

            if image_path and os.path.exists(image_path):
                _render_styled_table(
                    slide, table_data, left=0.81, top=1.80, width=6.75, height=content_h - 0.12
                )
                _render_framed_image_panel(
                    slide,
                    image_path=image_path,
                    panel_left=7.80,
                    panel_top=1.74,
                    panel_width=4.78,
                    panel_height=content_h,
                    caption_text="Visual Comparison Matrix generated via Gemini 2.5 Flash Image",
                    accent_color=ACCENT_EMERALD,
                    badge_label="COMPARISON INFOGRAPHIC",
                )
            else:
                _render_styled_table(
                    slide, table_data, left=0.81, top=1.80, width=11.71, height=content_h - 0.12
                )

            _render_bottom_takeaway_bar(slide, key_insight, ACCENT_EMERALD)

        # ======================================================================
        # LAYOUT 5: MULTI-CARD GRID + STRATEGIC INSIGHT PANEL ('cards' / 'text')
        # ======================================================================
        elif layout_type in ("cards", "text"):
            _render_executive_header(
                slide, title_text, subtitle_text, section_tag, idx, total_slides, ACCENT_GOLD
            )
            _render_feature_cards_column(
                slide, bullets, col_left=0.75, col_top=1.74, col_width=7.15, col_height=5.20
            )
            panel = slide.shapes.add_shape(
                MSO_SHAPE.ROUNDED_RECTANGLE,
                Inches(8.15), Inches(1.74), Inches(4.43), Inches(5.20)
            )
            panel.fill.solid()
            panel.fill.fore_color.rgb = RGBColor(22, 28, 54)
            panel.line.color.rgb = ACCENT_PURPLE
            panel.line.width = Pt(1.8)

            p_bar = slide.shapes.add_shape(
                MSO_SHAPE.RECTANGLE,
                Inches(8.15), Inches(1.74), Inches(4.43), Inches(0.09)
            )
            p_bar.fill.solid()
            p_bar.fill.fore_color.rgb = ACCENT_PURPLE
            p_bar.line.fill.background()

            ibadge = slide.shapes.add_shape(
                MSO_SHAPE.ROUNDED_RECTANGLE,
                Inches(8.45), Inches(2.05), Inches(2.60), Inches(0.38)
            )
            ibadge.fill.solid()
            ibadge.fill.fore_color.rgb = RGBColor(14, 20, 38)
            ibadge.line.color.rgb = ACCENT_CYAN
            ibadge.line.width = Pt(1.2)
            tf_ib = ibadge.text_frame
            tf_ib.margin_top = Inches(0.06)
            p_ib = tf_ib.paragraphs[0]
            p_ib.alignment = PP_ALIGN.CENTER
            p_ib.text = "STRATEGIC SYNTHESIS"
            p_ib.font.name = FONT_HEADING
            p_ib.font.size = Pt(9.5)
            p_ib.font.bold = True
            p_ib.font.color.rgb = ACCENT_CYAN

            ibox = slide.shapes.add_textbox(Inches(8.45), Inches(2.62), Inches(3.85), Inches(4.05))
            tf_i = ibox.text_frame
            tf_i.word_wrap = True
            p_i = tf_i.paragraphs[0]
            p_i.text = key_insight if key_insight else "Synthesizing innate speed with adaptive precision enables durable immunological protection."
            p_i.font.name = FONT_BODY
            p_i.font.size = Pt(14)
            p_i.font.bold = True
            p_i.font.color.rgb = TEXT_WHITE

        # ======================================================================
        # LAYOUT 6: CLOSING SUMMARY & Q&A SLIDE ('ending')
        # ======================================================================
        elif layout_type == "ending":
            _render_executive_header(
                slide, title_text, subtitle_text or "Executive Takeaways & Strategic Outlook", section_tag, idx, total_slides, ACCENT_GOLD
            )
            _render_feature_cards_column(
                slide, bullets, col_left=0.75, col_top=1.74, col_width=7.05, col_height=5.20
            )
            end_card = slide.shapes.add_shape(
                MSO_SHAPE.ROUNDED_RECTANGLE,
                Inches(8.05), Inches(1.74), Inches(4.53), Inches(5.20)
            )
            end_card.fill.solid()
            end_card.fill.fore_color.rgb = RGBColor(18, 28, 52)
            end_card.line.color.rgb = ACCENT_GOLD
            end_card.line.width = Pt(1.8)

            e_bar = slide.shapes.add_shape(
                MSO_SHAPE.RECTANGLE,
                Inches(8.05), Inches(1.74), Inches(4.53), Inches(0.09)
            )
            e_bar.fill.solid()
            e_bar.fill.fore_color.rgb = ACCENT_GOLD
            e_bar.line.fill.background()

            e_box = slide.shapes.add_textbox(Inches(8.35), Inches(2.05), Inches(3.95), Inches(4.65))
            tf_e = e_box.text_frame
            tf_e.word_wrap = True
            p_eh = tf_e.paragraphs[0]
            p_eh.text = "EXECUTIVE SUMMARY & NEXT STEPS"
            p_eh.font.name = FONT_HEADING
            p_eh.font.size = Pt(12)
            p_eh.font.bold = True
            p_eh.font.color.rgb = ACCENT_GOLD
            p_eh.space_after = Pt(14)

            p_eb = tf_e.add_paragraph()
            p_eb.text = key_insight if key_insight else "Thank you! Questions & Discussion."
            p_eb.font.name = FONT_BODY
            p_eb.font.size = Pt(14)
            p_eb.font.bold = True
            p_eb.font.color.rgb = TEXT_WHITE

    output_path = os.path.abspath(file_name)
    prs.save(output_path)
    print(f"Compiled Executive Presentation -> {output_path} ({total_slides} slides)")
    return f"Successfully compiled {total_slides}-slide executive presentation at: {output_path}" 

---
## 5. Step 4 Tool: Human-in-the-Loop Review Gate (`submit_presentation_for_review`)
Protected by `HumanInTheLoopMiddleware`. Whenever the agent calls `submit_presentation_for_review`, execution pauses and persists state in `SqliteSaver` (`ppt_agent_memory.db`).
- **Approve (`{"type": "approve"}`)**: Finalizes the presentation.
- **Reject with Feedback (`{"type": "reject", "message": "..."}`)**: Instructs the agent to revise the blueprint, generate additional Gemini visuals, and re-compile an enhanced `.pptx` deck!

In [6]:
@tool
def submit_presentation_for_review(
    pptx_file_path: str,
    slide_count: int,
    layout_breakdown: str,
    outline_summary: str,
) -> str:
    """
    Submits the compiled PowerPoint (.pptx) presentation for Human-in-the-Loop (HITL) review.
    Always call this tool after `create_premium_presentation` finishes saving the deck.
    """
    return (
        f"Presentation '{pptx_file_path}' ({slide_count} slides | Layouts: {layout_breakdown}) "
        f"has been approved by the user! Summary: {outline_summary}"
    )


# Setup Persistent SQLite Checkpointer
DB_PATH = os.path.join(os.getcwd(), "ppt_agent_memory.db")
sqlite_conn = sqlite3.connect(DB_PATH, check_same_thread=False)
memory = SqliteSaver(sqlite_conn)

# Configure HumanInTheLoopMiddleware on submit_presentation_for_review
hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "submit_presentation_for_review": {
            "allowed_decisions": ["approve", "edit", "reject"],
            "description": "Review the generated PowerPoint (.pptx) file. Approve if satisfied, or reject with enhancement feedback.",
        }
    },
    description_prefix="PPT Review & Enhancement Gate",
)

SYSTEM_PROMPT = """You are an Executive AI PowerPoint Architect & Scientific Visual Storyteller.

You MUST follow this strict 4-step workflow for every presentation request:

1. **LOAD SKILL (`load_skill`)**:
   - First call `load_skill("ppt_design")` to load the executive layout & typography specification.

2. **PLAN PRESENTATION BLUEPRINT FIRST (`plan_presentation_blueprint`)**:
   - Before generating ANY images or slides, call `plan_presentation_blueprint` to design the slide-by-slide blueprint.
   - **CRITICAL LAYOUT VARIETY RULE**: Do NOT place images only on the right side! Across a 6+ slide presentation, you MUST mix layouts:
     * Slide 1: `layout="title"` (`needs_image=False`, `image_position="none"`)
     * Content slides: Alternate between `layout="image_right"` (`image_position="right"`), `layout="image_left"` (`image_position="left"`), `layout="table"` (`image_position="table_companion"` with `needs_image=True` so a table comparison visual is generated alongside the native table!), and `layout="cards"` (`needs_image=False`).
     * Final slide: `layout="ending"` (`needs_image=False`, `image_position="none"`)

3. **GENERATE IMAGES ACCORDING TO BLUEPRINT (`generate_slide_image`)**:
   - Call `generate_slide_image` for every slide in your blueprint where `needs_image=True`.
   - Pass `visual_type="table_infographic"` for `"table"` slides and `visual_type="illustration"` for `"image_right"` / `"image_left"` slides.

4. **COMPILE EXECUTIVE PPTX (`create_premium_presentation`) & SUBMIT (`submit_presentation_for_review`)**:
   - Format every bullet point as `'Bold Concept Title: Crisp, detailed executive explanation...'` so each bullet renders as a structured two-tier Feature Card with numbered badges and neon accent bars.
   - For `layout="table"` slides, always populate `table_data={"headers": [...], "rows": [[...], ...]}` AND pass the `image_path` generated for that table slide!
   - Always include a rich `key_insight` string on every content and ending slide.
   - Finally, call `submit_presentation_for_review`. If the user rejects with enhancement feedback, update the blueprint/images/slides and re-compile!
"""

ppt_agent = create_agent(
    model=model,
    tools=[
        load_skill,
        plan_presentation_blueprint,
        generate_slide_image,
        create_premium_presentation,
        submit_presentation_for_review,
    ],
    system_prompt=SYSTEM_PROMPT,
    middleware=[hitl_middleware],
    checkpointer=memory,
)
print("Executive PPT Agent initialized with Plan-First Workflow, 6 Dynamic Layouts, HITL Middleware & SqliteSaver!")

Executive PPT Agent initialized with Plan-First Workflow, 6 Dynamic Layouts, HITL Middleware & SqliteSaver!


---
## 6. End-to-End Execution — Draft 1 (6 Slides: `title`, `image_right`, `image_left`, `table`, `cards`, `ending`)
Let's run our upgraded agent to create a **6-slide presentation on the Human Immune System** (`Human_Immune_System_v1_Draft.pptx`).
Watch how the agent:
1. Loads `ppt_design` skill
2. Calls `plan_presentation_blueprint` first (planning `title`, `image_right`, `image_left`, `table`, `cards`, `ending`)
3. Generates 3 targeted Gemini visuals (`slide_2_illustration.png` for `image_right`, `slide_3_illustration.png` for `image_left`, and `slide_4_table_infographic.png` for `table`)
4. Compiles `Human_Immune_System_v1_Draft.pptx` with executive cards, left/right visuals, and a styled comparison table
5. Pauses at `HumanInTheLoopMiddleware` (`submit_presentation_for_review`)

In [7]:
thread_config = {"configurable": {"thread_id": "immune_system_executive_v2"}}

user_request = (
    "Create a 6-slide executive presentation on 'The Human Immune System: Innate vs. Adaptive Defense Architecture' "
    "saved as 'Human_Immune_System_v1_Draft.pptx'. "
    "Make sure you plan the presentation first using `plan_presentation_blueprint` with varied layouts: "
    "Slide 1: 'title', "
    "Slide 2: 'image_right' (Innate Immunity First-Line Defense with image on the right), "
    "Slide 3: 'image_left' (Adaptive Immunity T-Cells & B-Cells with image on the left), "
    "Slide 4: 'table' (Innate vs. Adaptive Comparison Table with both native table_data and a table_infographic image), "
    "Slide 5: 'cards' (Immunological Memory & Vaccine Mechanisms multi-card breakdown), "
    "Slide 6: 'ending' (Executive Summary & Clinical Outlook)."
)

response_v1 = ppt_agent.invoke(
    {"messages": [{"role": "user", "content": user_request}]},
    config=thread_config,
)

state_v1 = ppt_agent.get_state(thread_config)
print("\n--- Agent Paused at HumanInTheLoopMiddleware? ---", bool(state_v1.next))
if state_v1.tasks and state_v1.tasks[0].interrupts:
    interrupt_payload = state_v1.tasks[0].interrupts[0].value
    print("Interrupt Action Requests:", json.dumps(interrupt_payload.get("action_requests", []), indent=2))

Loaded skill 'ppt_design' (6,933 chars) from /Users/sachinmishra/Desktop/Agents_From_Scratch/ppt-agent/skills/ppt_design/SKILL.md

PRESENTATION BLUEPRINT LOCKED: The Human Immune System: Innate vs. Adaptive Defense Architecture (6 Slides)
  • Slide 1: [TITLE] The Human Immune System: Innate vs. Adaptive Defense Architecture (Image: none)
  • Slide 2: [IMAGE_RIGHT] Innate Immunity: Our First Line of Defense (Image: right)
  • Slide 3: [IMAGE_LEFT] Adaptive Immunity: Targeted & Memory-Driven Response (Image: left)
  • Slide 4: [TABLE] Innate vs. Adaptive: Key Differentiators (Image: table_companion)
  • Slide 5: [CARDS] Immunological Memory & Vaccine Mechanisms (Image: none)
  • Slide 6: [ENDING] Executive Summary & Clinical Outlook (Image: none)
  -> Planned Gemini Visuals to Generate: 3 slide(s)

Generating Gemini 2.5 Flash Image for Slide 2 (illustration)...
Saved Slide 2 image -> /Users/sachinmishra/Desktop/Agents_From_Scratch/ppt-agent/generated_Slide_Images/slide_2_illustration.png

---
## 7. Resuming from Human-in-the-Loop Middleware with Enhancement Feedback (Draft 2 -> 8 Slides)
Now we simulate inspecting `Human_Immune_System_v1_Draft.pptx` and sending a **`reject` decision with enhancement instructions** via `Command(resume=...)`:
- Expand the presentation to **8 slides** saved as `Human_Immune_System_v2_Enhanced.pptx`
- Add a new **Slide 5 (`layout="image_left"`)** on **mRNA Lipid Nanoparticle Vaccine Architecture** with a custom Gemini illustration on the **left**
- Add a new **Slide 6 (`layout="image_right"`)** on **CAR-T Cell Cancer Immunotherapy** with a custom Gemini illustration on the **right**

In [8]:
human_feedback = (
    "The 6-slide draft looks great! Please enhance the presentation into an 8-slide deck saved as "
    "'Human_Immune_System_v2_Enhanced.pptx' by inserting two new visual slides before the cards and ending slides: "
    "1) Slide 5 with layout='image_left' covering 'mRNA Vaccine & Lipid Nanoparticle Delivery' (generate a new Gemini illustration for Slide 5 on the left side), and "
    "2) Slide 6 with layout='image_right' covering 'CAR-T Cell Engineered Immunotherapy' (generate a new Gemini illustration for Slide 6 on the right side). "
    "Keep existing Slides 1-4 (including the Slide 4 comparison table) and shift the Cards slide to Slide 7 and Ending slide to Slide 8. "
    "Update the blueprint first, generate the 2 new images, compile 'Human_Immune_System_v2_Enhanced.pptx', and submit for review."
)

response_v2 = ppt_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": human_feedback,
                }
            ]
        }
    ),
    config=thread_config,
)

state_v2 = ppt_agent.get_state(thread_config)
print("\n--- Agent Paused at HumanInTheLoopMiddleware for V2 Review? ---", bool(state_v2.next))
if state_v2.tasks and state_v2.tasks[0].interrupts:
    interrupt_payload_v2 = state_v2.tasks[0].interrupts[0].value
    print("V2 Interrupt Action Requests:", json.dumps(interrupt_payload_v2.get("action_requests", []), indent=2))


PRESENTATION BLUEPRINT LOCKED: The Human Immune System: Innate vs. Adaptive Defense Architecture (8 Slides)
  • Slide 1: [TITLE] The Human Immune System: Innate vs. Adaptive Defense Architecture (Image: none)
  • Slide 2: [IMAGE_RIGHT] Innate Immunity: Our First Line of Defense (Image: right)
  • Slide 3: [IMAGE_LEFT] Adaptive Immunity: Targeted & Memory-Driven Response (Image: left)
  • Slide 4: [TABLE] Innate vs. Adaptive: Key Differentiators (Image: table_companion)
  • Slide 5: [IMAGE_LEFT] mRNA Vaccines: Precision Delivery & Immune Activation (Image: left)
  • Slide 6: [IMAGE_RIGHT] CAR-T Cell Therapy: Engineering Immunity Against Cancer (Image: right)
  • Slide 7: [CARDS] Immunological Memory & Vaccine Principles (Image: none)
  • Slide 8: [ENDING] Executive Summary & Clinical Outlook (Image: none)
  -> Planned Gemini Visuals to Generate: 5 slide(s)

Generating Gemini 2.5 Flash Image for Slide 5 (illustration)...
Saved Slide 5 image -> /Users/sachinmishra/Desktop/Agents_From_Scr

---
## 8. Final Human-in-the-Loop Approval (`type="approve"`)
Once satisfied with `Human_Immune_System_v2_Enhanced.pptx`, we send `{"type": "approve"}` to complete the workflow and persist the final state in SQLite.

In [9]:
final_response = ppt_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=thread_config,
)
print("\n=== FINAL AGENT RESPONSE ===")
print(final_response["messages"][-1].content)


=== FINAL AGENT RESPONSE ===
[{'type': 'text', 'text': 'The 8-slide executive presentation "Human_Immune_System_v2_Enhanced.pptx" has been successfully created and approved. It covers:\n\n*   **Slide 1: Introduction** - The Human Immune System: Innate vs. Adaptive Defense Architecture\n*   **Slide 2: Innate Immunity** - Our First Line of Defense (Image Right)\n*   **Slide 3: Adaptive Immunity** - Targeted & Memory-Driven Response (Image Left)\n*   **Slide 4: Comparison** - Innate vs. Adaptive: Key Differentiators (Table with Infographic)\n*   **Slide 5: Vaccine Innovation** - mRNA Vaccines: Precision Delivery & Immune Activation (Image Left)\n*   **Slide 6: Immunotherapy Frontiers** - CAR-T Cell Therapy: Engineering Immunity Against Cancer (Image Right)\n*   **Slide 7: Immunological Memory** - Immunological Memory & Vaccine Principles (Cards Layout)\n*   **Slide 8: Conclusion** - Executive Summary & Clinical Outlook (Ending Layout)', 'extras': {'signature': 'Cm8BaRR9E26tA5jGXviVo52WLi